<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LHM_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕺 LHM & LHM++ — Large Animatable Human Reconstruction Model (3D Gaussian Avatar + SMPL-X Animation)
**One-Click Google Colab Notebook** • [aigc3d/LHM-plusplus](https://github.com/aigc3d/LHM-plusplus) • [aigc3d/LHM](https://github.com/aigc3d/LHM) • [3DAIGC/LHMPP-700M](https://huggingface.co/3DAIGC/LHMPP-700M) • [3DAIGC/LHM-1B](https://huggingface.co/3DAIGC/LHM-1B)

Reconstruct **animatable 3D Gaussian Splatting (`.ply`) human avatars** and render **pose-driven animated character videos (`.mp4`)** in under a second from **1 to 8 uncalibrated reference photos** using Alibaba XR Lab's **LHM++** (`3DAIGC/LHMPP-700M`, ICCV 2025 Highlight / March 2026 release) and **LHM** (`3DAIGC/LHM-1B` / `LHM-1B-HF` / `LHM-MINI`).

### ✨ Key Capabilities
* **Multi-View & Single-View Animatable 3D Avatar Reconstruction:**
  * **LHM++ (`3DAIGC/LHMPP-700M`, `LHMPP-700M-PixelShuffle`, `LHMPP-700M-SMPLX-FREE`):** Accepts **1 to 8 uncalibrated reference photos** of a character, runs feed-forward reconstruction in **~0.8–1.3s** on just **8 GB VRAM** (Colab T4 / L4 / A100), and supports both neural decoder rendering (`neural_render`) and direct 3D Gaussian Splatting rasterization (`gs_render`).
  * **LHM (`3DAIGC/LHM-1B`, `3DAIGC/LHM-1B-HF`, `3DAIGC/LHM-500M-HF`, `3DAIGC/LHM-MINI`):** Feed-forward single-view full-body / half-body Multimodal Transformer + Head Token + 3DGS avatar reconstruction.
* **SMPL-X Pose-Driven Character Animation (`.mp4`):** Drive reconstructed avatars with built-in SMPL-X motion sequences (`mimo1`, `mimo2`, `mimo4`, `mimo6`, dance, martial arts, sports) or upload your own custom SMPL-X pose sequence directory / driving video.
* **Canonical T-Pose & Posed 3D Gaussian Splatting Export (`.ply` + `.glb` Preview):** Export both the canonical neutral-pose 3DGS `.ply` avatar (`run_tpose_export` / `inference_gs` / `animation_infer_gs`) and any posed animation frame `.ply`, plus an interactive 3D point-cloud `.glb` preview right in the Gradio UI.
* **Google Drive Weight & Prior Caching:** Stores all Hugging Face / ModelScope weights, SMPL-X human model priors (`3DAIGC/LHMPP-Prior`), and motion sequences (`3DAIGC/LHMPP-Assets`) under `/content/drive/MyDrive/AEI_3D_Cache/LHM`.

### 🖥️ Recommended Runtime
| GPU Runtime | VRAM | Recommended Model | Typical Reconstruction Latency |
| :--- | :--- | :--- | :--- |
| **Colab Free T4** | 15 GB | `3DAIGC/LHMPP-700M` or `3DAIGC/LHM-MINI` | ~1.2s avatar reconstruction + ~25 FPS animation |
| **Colab Pro L4** | 22.5 GB | `3DAIGC/LHMPP-700M`, `LHMPP-700M-PS`, or `3DAIGC/LHM-1B-HF` | ~0.9s avatar reconstruction + ~35 FPS animation |
| **Colab Pro A100** | 40 / 80 GB | `3DAIGC/LHMPP-700M` or `3DAIGC/LHM-1B` | ~0.6s avatar reconstruction + ~50 FPS animation |

In [ ]:
# @title ⚙️ Step 1: Environment Setup (Mount Drive, Clone LHM++ & LHM, Install 3DGS & SMPL-X Stack)
# @markdown Mounts Google Drive, configures `HF_HOME` to `/content/drive/MyDrive/AEI_3D_Cache/LHM/hf_cache`, clones both `aigc3d/LHM-plusplus` and `aigc3d/LHM`, and installs the 3D Gaussian Splatting (`diff-gaussian-rasterization`, `simple-knn`, `gsplat`), `pointops`, and SMPL-X dependency stack.
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/AEI_3D_Cache/LHM"  # @param {type:"string"}
INSTALL_CUDA_3DGS_EXTENSIONS = True  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import shutil
import subprocess
from pathlib import Path


def run_cmd(cmd, cwd=None, check=False):
    print(f"\n$ {cmd}")
    res = subprocess.run(cmd, shell=True, cwd=cwd, text=True)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {res.returncode}: {cmd}")
    return res.returncode


# 1. Mount Google Drive & Set HF_HOME Before Any Imports
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
        print(f"✅ Google Drive mounted. Cache directory: {DRIVE_CACHE_DIR}")
    except Exception as _e:
        print(f"⚠️ Google Drive mount skipped or unavailable ({_e}). Using local cache.")
        DRIVE_CACHE_DIR = "/content/cache/LHM"
        os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
else:
    DRIVE_CACHE_DIR = "/content/cache/LHM"
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

HF_CACHE_DIR = os.path.join(DRIVE_CACHE_DIR, "hf_cache")
MODELSCOPE_CACHE_DIR = os.path.join(DRIVE_CACHE_DIR, "modelscope_cache")
OUTPUT_BASE_DIR = os.path.join(DRIVE_CACHE_DIR, "outputs")
for _d in (HF_CACHE_DIR, MODELSCOPE_CACHE_DIR, OUTPUT_BASE_DIR):
    os.makedirs(_d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TORCH_HOME"] = os.path.join(DRIVE_CACHE_DIR, "torch_cache")
os.environ["MODELSCOPE_CACHE"] = MODELSCOPE_CACHE_DIR
os.environ["NUMBA_THREADING_LAYER"] = "omp"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

# 2. Verify GPU & CUDA Architecture
import torch

if not torch.cuda.is_available():
    print("⚠️ WARNING: No CUDA GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 / L4 / A100).")
    GPU_NAME = "CPU"
    CC_MAJOR, CC_MINOR = (0, 0)
else:
    GPU_NAME = torch.cuda.get_device_name(0)
    CC_MAJOR, CC_MINOR = torch.cuda.get_device_capability(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ Detected GPU: {GPU_NAME} ({VRAM_GB:.1f} GB VRAM, Compute Capability {CC_MAJOR}.{CC_MINOR})")
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC_MAJOR}.{CC_MINOR}"

# 3. Clone Official aigc3d/LHM-plusplus and aigc3d/LHM Repositories
LHMPP_REPO_DIR = "/content/LHM-plusplus"
LHM_REPO_DIR = "/content/LHM"

if not os.path.exists(os.path.join(LHMPP_REPO_DIR, "app.py")):
    run_cmd(f"git clone --depth 1 https://github.com/aigc3d/LHM-plusplus.git {LHMPP_REPO_DIR}", check=True)
else:
    print(f"✅ LHM++ repository already exists at {LHMPP_REPO_DIR}")

if not os.path.exists(os.path.join(LHM_REPO_DIR, "app.py")):
    run_cmd(f"git clone --depth 1 https://github.com/aigc3d/LHM.git {LHM_REPO_DIR}", check=True)
else:
    print(f"✅ LHM repository already exists at {LHM_REPO_DIR}")

# 4. Install Python Dependencies
run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 ninja-build")

PIP_PACKAGES = [
    "gradio==5.49.1",
    "tqdm==4.66.5",
    "huggingface_hub>=0.26.0",
    "modelscope",
    "safetensors",
    "accelerate",
    "omegaconf",
    "einops",
    "roma",
    "smplx",
    "chumpy",
    "plyfile",
    "trimesh",
    "rembg[gpu]",
    "onnxruntime-gpu",
    "imageio",
    "imageio-ffmpeg",
    "ffmpegio",
    "decord",
    "kornia",
    "timm",
    "lpips",
    "kiui",
    "loguru",
    "megfile",
    "av",
    " opencv-python-headless".strip(),
    "scikit-image",
    "scipy",
    "matplotlib",
    "ultralytics",
    "open3d",
]
run_cmd(f"{sys.executable} -m pip install -q --upgrade " + " ".join(f'"{p}"' for p in PIP_PACKAGES), check=True)

# 5. Install 3D Gaussian Splatting & PointOps CUDA Extensions
if INSTALL_CUDA_3DGS_EXTENSIONS and torch.cuda.is_available():
    print("🔧 Checking / installing 3DGS rasterizer, simple-knn, gsplat, and pointops...")
    run_cmd(f"{sys.executable} -m pip install -q gsplat spconv-cu120 || {sys.executable} -m pip install -q gsplat")
    try:
        import diff_gaussian_rasterization  # noqa: F401
        print("✅ diff_gaussian_rasterization already installed.")
    except Exception as _e:
        run_cmd(
            f"{sys.executable} -m pip install -q --no-build-isolation "
            f'"git+https://github.com/ashawkey/diff-gaussian-rasterization.git"'
        )
    try:
        import simple_knn  # noqa: F401
        print("✅ simple_knn already installed.")
    except Exception as _e:
        run_cmd(
            f"{sys.executable} -m pip install -q --no-build-isolation "
            f'"git+https://github.com/camenduru/simple-knn.git"'
        )
    pointops_dir = os.path.join(LHMPP_REPO_DIR, "extensions", "pointops")
    if os.path.isdir(pointops_dir):
        try:
            import pointops  # noqa: F401
            print("✅ pointops already installed.")
        except Exception as _e:
            run_cmd(f"{sys.executable} -m pip install -q --no-build-isolation {pointops_dir}")

if LHMPP_REPO_DIR not in sys.path:
    sys.path.insert(0, LHMPP_REPO_DIR)

print("\n🎉 Step 1 complete! Proceed to Step 2 to download & cache LHM++ / LHM weights and SMPL-X priors.")

In [ ]:
# @title 📦 Step 2: Download & Cache LHM++ / LHM Weights, SMPL-X Priors & Motion Assets to Google Drive
# @markdown Downloads and links the selected **LHM++** (`3DAIGC/LHMPP-700M`, `Damo_XR_Lab/LHMPP-700M-PixelShuffle`, `Damo_XR_Lab/LHMPP-700M-SMPLX-FREE`) or **LHM** (`3DAIGC/LHM-1B`, `3DAIGC/LHM-1B-HF`, `3DAIGC/LHM-500M-HF`, `3DAIGC/LHM-MINI`) checkpoint, plus the SMPL-X / BiRefNet / VGGHead prior models (`3DAIGC/LHMPP-Prior`) and driving motion library (`3DAIGC/LHMPP-Assets`), into `/content/drive/MyDrive/AEI_3D_Cache/LHM`.
PRIMARY_MODEL_ID = "3DAIGC/LHMPP-700M"  # @param ["3DAIGC/LHMPP-700M", "Damo_XR_Lab/LHMPP-700M-PixelShuffle", "Damo_XR_Lab/LHMPP-700M-SMPLX-FREE", "3DAIGC/LHM-1B", "3DAIGC/LHM-1B-HF", "3DAIGC/LHM-500M-HF", "3DAIGC/LHM-MINI"]
ALSO_CACHE_LHM_1B = False  # @param {type:"boolean"}
CACHE_PRIORS_AND_MOTIONS = True  # @param {type:"boolean"}
FORCE_REDOWNLOAD = False  # @param {type:"boolean"}

import os
import shutil
import tarfile
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")

PRIOR_DRIVE_DIR = os.path.join(DRIVE_CACHE_DIR, "pretrained_models")
ASSETS_DRIVE_DIR = os.path.join(DRIVE_CACHE_DIR, "assets")
MOTION_DRIVE_DIR = os.path.join(DRIVE_CACHE_DIR, "motion_video")
os.makedirs(PRIOR_DRIVE_DIR, exist_ok=True)
os.makedirs(ASSETS_DRIVE_DIR, exist_ok=True)
os.makedirs(MOTION_DRIVE_DIR, exist_ok=True)

MODEL_REGISTRY = {
    "3DAIGC/LHMPP-700M": {
        "family": "lhmpp",
        "card_key": "LHMPP-700M",
        "hf_repo": "3DAIGC/LHMPP-700M",
        "ms_repo": "Damo_XR_Lab/LHMPP-700M",
        "config": "/content/LHM-plusplus/configs/inference/human-lrm-700M.yaml",
    },
    "Damo_XR_Lab/LHMPP-700M-PixelShuffle": {
        "family": "lhmpp",
        "card_key": "LHMPP-700M-PS",
        "hf_repo": None,
        "ms_repo": "Damo_XR_Lab/LHMPP-700M-PixelShuffle",
        "config": "/content/LHM-plusplus/configs/inference/human-lrm-700M-PS.yaml",
    },
    "Damo_XR_Lab/LHMPP-700M-SMPLX-FREE": {
        "family": "lhmpp",
        "card_key": "LHMPP-700M-SF",
        "hf_repo": None,
        "ms_repo": "Damo_XR_Lab/LHMPP-700M-SMPLX-FREE",
        "config": "/content/LHM-plusplus/configs/inference/human-lrm-700M.yaml",
    },
    "3DAIGC/LHM-1B": {
        "family": "lhm",
        "card_key": "LHM-1B",
        "hf_repo": "3DAIGC/LHM-1B",
        "ms_repo": "Damo_XR_Lab/LHM-1B",
        "config": "/content/LHM/configs/inference/human-lrm-1B.yaml",
    },
    "3DAIGC/LHM-1B-HF": {
        "family": "lhm",
        "card_key": "LHM-1B-HF",
        "hf_repo": "3DAIGC/LHM-1B-HF",
        "ms_repo": "Damo_XR_Lab/LHM-1B-HF",
        "config": "/content/LHM/configs/inference/human-lrm-1B.yaml",
    },
    "3DAIGC/LHM-500M-HF": {
        "family": "lhm",
        "card_key": "LHM-500M-HF",
        "hf_repo": "3DAIGC/LHM-500M-HF",
        "ms_repo": "Damo_XR_Lab/LHM-500M-HF",
        "config": "/content/LHM/configs/inference/human-lrm-500M.yaml",
    },
    "3DAIGC/LHM-MINI": {
        "family": "lhm",
        "card_key": "LHM-MINI",
        "hf_repo": "3DAIGC/LHM-MINI",
        "ms_repo": "Damo_XR_Lab/LHM-MINI",
        "config": "/content/LHM/configs/inference/human-lrm-mini.yaml",
    },
}


def ensure_symlink_or_dir(src_dir: str, dst_link: str):
    os.makedirs(src_dir, exist_ok=True)
    if os.path.islink(dst_link):
        os.unlink(dst_link)
    elif os.path.isdir(dst_link):
        for item in os.listdir(dst_link):
            s = os.path.join(dst_link, item)
            d = os.path.join(src_dir, item)
            if not os.path.exists(d):
                shutil.move(s, d)
        shutil.rmtree(dst_link)
    os.makedirs(os.path.dirname(os.path.abspath(dst_link)), exist_ok=True)
    os.symlink(src_dir, dst_link)


def cache_priors_and_assets(force: bool = False):
    print("📥 Caching SMPL-X / VGGHead / BiRefNet prior models (3DAIGC/LHMPP-Prior)...")
    try:
        snapshot_download(
            repo_id="3DAIGC/LHMPP-Prior",
            local_dir=PRIOR_DRIVE_DIR,
            force_download=force,
        )
        print(f"✅ Cached 3DAIGC/LHMPP-Prior -> {PRIOR_DRIVE_DIR}")
    except Exception as e:
        print(f"⚠️ Hugging Face prior download warning ({e}), falling back to ModelScope...")
        from modelscope import snapshot_download as ms_snapshot_download
        ms_snapshot_download(model_id="Damo_XR_Lab/LHMPP-Prior", local_dir=PRIOR_DRIVE_DIR)

    print("📥 Caching example character views & SMPL-X driving motions (3DAIGC/LHMPP-Assets)...")
    try:
        snapshot_download(
            repo_id="3DAIGC/LHMPP-Assets",
            local_dir=ASSETS_DRIVE_DIR,
            force_download=force,
        )
        print(f"✅ Cached 3DAIGC/LHMPP-Assets -> {ASSETS_DRIVE_DIR}")
    except Exception as e:
        print(f"⚠️ Hugging Face assets download warning ({e}), falling back to ModelScope...")
        from modelscope import snapshot_download as ms_snapshot_download
        ms_snapshot_download(model_id="Damo_XR_Lab/LHMPP-Assets", local_dir=ASSETS_DRIVE_DIR)

    # Extract motion_video.tar if present
    tar_path = os.path.join(ASSETS_DRIVE_DIR, "motion_video.tar")
    if os.path.isfile(tar_path) and not os.listdir(MOTION_DRIVE_DIR):
        print(f"📦 Extracting {tar_path} into {DRIVE_CACHE_DIR} ...")
        with tarfile.open(tar_path, "r") as tar:
            tar.extractall(path=DRIVE_CACHE_DIR)
        print(f"✅ Extracted SMPL-X motion sequences -> {MOTION_DRIVE_DIR}")

    # Link into both LHM-plusplus and LHM workspaces
    for repo_root in (LHMPP_REPO_DIR, LHM_REPO_DIR):
        if os.path.isdir(repo_root):
            ensure_symlink_or_dir(PRIOR_DRIVE_DIR, os.path.join(repo_root, "pretrained_models"))
            ensure_symlink_or_dir(MOTION_DRIVE_DIR, os.path.join(repo_root, "motion_video"))
            train_data_dir = os.path.join(repo_root, "train_data")
            os.makedirs(train_data_dir, exist_ok=True)
            ensure_symlink_or_dir(MOTION_DRIVE_DIR, os.path.join(train_data_dir, "motion_video"))

    # Copy assets subdirectories into LHM-plusplus/assets
    lhmpp_assets = os.path.join(LHMPP_REPO_DIR, "assets")
    os.makedirs(lhmpp_assets, exist_ok=True)
    for item in os.listdir(ASSETS_DRIVE_DIR):
        if item == "motion_video.tar":
            continue
        s = os.path.join(ASSETS_DRIVE_DIR, item)
        d = os.path.join(lhmpp_assets, item)
        if not os.path.exists(d):
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d)


def cache_lhm_checkpoint(model_id: str, force: bool = False) -> str:
    info = MODEL_REGISTRY[model_id]
    target_dir = os.path.join(PRIOR_DRIVE_DIR, "huggingface", info["card_key"])
    os.makedirs(target_dir, exist_ok=True)
    print(f"📥 Caching checkpoint '{model_id}' -> {target_dir} ...")
    if info["hf_repo"]:
        try:
            local_dir = snapshot_download(
                repo_id=info["hf_repo"],
                local_dir=target_dir,
                cache_dir=os.path.join(HF_CACHE_DIR, "hub"),
                force_download=force,
            )
            print(f"✅ Cached {model_id} at {local_dir}")
            return local_dir
        except Exception as e:
            print(f"⚠️ Hugging Face download warning for {model_id} ({e}), trying ModelScope...")
    from modelscope import snapshot_download as ms_snapshot_download
    local_dir = ms_snapshot_download(model_id=info["ms_repo"], local_dir=target_dir)
    print(f"✅ Cached {model_id} via ModelScope at {local_dir}")
    return local_dir


if CACHE_PRIORS_AND_MOTIONS:
    cache_priors_and_assets(force=FORCE_REDOWNLOAD)

CACHED_MODEL_PATHS = {}
CACHED_MODEL_PATHS[PRIMARY_MODEL_ID] = cache_lhm_checkpoint(PRIMARY_MODEL_ID, force=FORCE_REDOWNLOAD)

if ALSO_CACHE_LHM_1B and PRIMARY_MODEL_ID != "3DAIGC/LHM-1B":
    CACHED_MODEL_PATHS["3DAIGC/LHM-1B"] = cache_lhm_checkpoint("3DAIGC/LHM-1B", force=FORCE_REDOWNLOAD)

print("\n🎉 Step 2 complete! Models, SMPL-X priors, and motion sequences are cached and linked.")

In [ ]:
# @title 🧠 Step 3: Core LHM++ & LHM 3D Gaussian Avatar Reconstruction & SMPL-X Animation Engine
# @markdown Defines the unified inference engine supporting both **LHM++ (`3DAIGC/LHMPP-700M`, 1–8 views)** and **LHM (`3DAIGC/LHM-1B`, single view)**, including:
# @markdown - **Multi-View / Single-View Reference Preprocessing** (BiRefNet / SAM2 foreground segmentation, VGGHeadDetector face crop, SMPL-X shape estimation)
# @markdown - **Canonical T-Pose & Posed 3D Gaussian Avatar Export (`.ply`)** (`run_tpose_export` / `inference_gs` / `animation_infer_gs` $\rightarrow$ `save_ply`)
# @markdown - **SMPL-X Driven Animated Character Video Rendering (`.mp4`)** (`inference_results` with `neural_render` or `gs_render`, and `animation_infer`)
# @markdown - **Interactive 3D `.glb` Point-Cloud Preview** converted directly from the exported 3DGS `.ply` avatar
import gc
import os
import sys
import time
import glob
import shutil
import tempfile
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any

import cv2
import torch
import trimesh
import numpy as np
from PIL import Image
from plyfile import PlyData
from omegaconf import OmegaConf

_ENGINE_STATE: Dict[str, Any] = {
    "model_id": None,
    "family": None,
    "lhm": None,
    "cfg": None,
    "pose_estimator": None,
    "facedetector": None,
    "parsingnet": None,
}


def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def list_available_motions() -> Dict[str, str]:
    """Discovers all built-in SMPL-X motion sequences in MOTION_DRIVE_DIR."""
    motions: Dict[str, str] = {}
    search_roots = [
        MOTION_DRIVE_DIR,
        os.path.join(LHMPP_REPO_DIR, "motion_video"),
        os.path.join(LHM_REPO_DIR, "train_data", "motion_video"),
    ]
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for entry in sorted(os.listdir(root)):
            entry_dir = os.path.join(root, entry)
            if not os.path.isdir(entry_dir):
                continue
            smplx_dir = os.path.join(entry_dir, "smplx_params")
            if os.path.isdir(smplx_dir) and entry not in motions:
                motions[entry] = smplx_dir
    return motions


def list_example_character_cases() -> Dict[str, List[str]]:
    """Discovers example single-view and multi-view character image sets from LHM++ / LHM assets."""
    cases: Dict[str, List[str]] = {}
    multi_root = os.path.join(LHMPP_REPO_DIR, "assets", "example_multi_images")
    if os.path.isdir(multi_root):
        for case_name in sorted(os.listdir(multi_root)):
            case_dir = os.path.join(multi_root, case_name)
            if os.path.isdir(case_dir):
                imgs = sorted(
                    os.path.join(case_dir, f)
                    for f in os.listdir(case_dir)
                    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
                )
                if imgs:
                    cases[f"Multi-View ({len(imgs)} views): {case_name}"] = imgs
    single_root = os.path.join(LHM_REPO_DIR, "train_data", "example_imgs")
    if os.path.isdir(single_root):
        for f in sorted(os.listdir(single_root))[:8]:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
                stem = os.path.splitext(f)[0]
                cases[f"Single-View: {stem}"] = [os.path.join(single_root, f)]
    return cases


def ply_to_glb_preview(ply_path: str, glb_path: str, max_points: int = 50_000) -> str:
    """Converts a 3D Gaussian Splatting .ply avatar into a colored .glb point cloud for Gradio 3D preview."""
    plydata = PlyData.read(ply_path)
    v = plydata["vertex"]
    xyz = np.stack([np.asarray(v["x"]), np.asarray(v["y"]), np.asarray(v["z"])], axis=-1).astype(np.float32)

    # Filter out extreme outlier splats or very low opacity splats
    valid_mask = np.isfinite(xyz).all(axis=-1)
    if "opacity" in v.data.dtype.names:
        opacities = 1.0 / (1.0 + np.exp(-np.asarray(v["opacity"], dtype=np.float32)))
        valid_mask &= opacities > 0.05
    xyz = xyz[valid_mask]

    C0 = 0.28209479177387814
    if all(k in v.data.dtype.names for k in ("f_dc_0", "f_dc_1", "f_dc_2")):
        dc = np.stack(
            [np.asarray(v["f_dc_0"]), np.asarray(v["f_dc_1"]), np.asarray(v["f_dc_2"])],
            axis=-1,
        ).astype(np.float32)[valid_mask]
        rgb = np.clip(0.5 + C0 * dc, 0.0, 1.0)
        rgba = np.concatenate([ (rgb * 255.0).astype(np.uint8), np.full((len(rgb), 1), 255, dtype=np.uint8) ], axis=-1)
    else:
        rgba = np.full((len(xyz), 4), [200, 210, 235, 255], dtype=np.uint8)

    if len(xyz) > max_points:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(xyz), size=max_points, replace=False)
        xyz = xyz[idx]
        rgba = rgba[idx]

    # Center and orient upright for standard 3D viewer
    if len(xyz) > 0:
        xyz = xyz - np.median(xyz, axis=0, keepdims=True)
        xyz[:, 1] = -xyz[:, 1]

    cloud = trimesh.PointCloud(vertices=xyz, colors=rgba)
    scene = trimesh.Scene([cloud])
    scene.export(glb_path)
    return glb_path


def load_lhm_engine(model_id: str = PRIMARY_MODEL_ID):
    """Loads and caches the requested LHM++ or LHM model and shared preprocessing modules on GPU."""
    if _ENGINE_STATE["model_id"] == model_id and _ENGINE_STATE["lhm"] is not None:
        return _ENGINE_STATE

    if _ENGINE_STATE["lhm"] is not None:
        print(f"🔄 Unloading previous model ({_ENGINE_STATE['model_id']})...")
        for k in ("lhm", "cfg"):
            _ENGINE_STATE[k] = None
        clear_vram()

    info = MODEL_REGISTRY[model_id]
    family = info["family"]
    local_ckpt_dir = cache_lhm_checkpoint(model_id, force=False)

    # Ensure active repo is at front of sys.path and cwd is set appropriately
    target_repo = LHMPP_REPO_DIR if family == "lhmpp" else LHM_REPO_DIR
    os.chdir(target_repo)
    if target_repo in sys.path:
        sys.path.remove(target_repo)
    sys.path.insert(0, target_repo)

    t0 = time.perf_counter()
    if family == "lhmpp":
        from core.utils.app_utils import parse_app_configs, build_app_model
        from core.utils.face_detector import VGGHeadDetector
        from engine.pose_estimation.pose_estimator import PoseEstimator
        from engine.SegmentAPI.BiRefNet import BiRefNetSeg

        os.environ["APP_ENABLED"] = "1"
        os.environ["APP_MODEL_NAME"] = local_ckpt_dir
        os.environ["APP_TYPE"] = "infer.human_lrm_a4o"

        cfg, _cfg_train = parse_app_configs(model_name=info["card_key"])
        cfg.model_name = local_ckpt_dir
        lhm = build_app_model(cfg)
        lhm.to("cuda")

        if _ENGINE_STATE["facedetector"] is None:
            fd = VGGHeadDetector("./pretrained_models/gagatracker/vgghead/vgg_heads_l.trcd", device="cuda")
            fd.to("cuda")
            _ENGINE_STATE["facedetector"] = fd
        if _ENGINE_STATE["pose_estimator"] is None:
            pe = PoseEstimator("./pretrained_models/human_model_files/", device="cpu")
            pe.to("cuda")
            pe.device = "cuda"
            _ENGINE_STATE["pose_estimator"] = pe
        if _ENGINE_STATE["parsingnet"] is None:
            _ENGINE_STATE["parsingnet"] = BiRefNetSeg(device="cuda")
    else:
        from LHM.utils.hf_hub import wrap_model_hub
        from LHM.models import model_dict
        from LHM.utils.face_detector import VGGHeadDetector as LHM_VGGHeadDetector
        from engine.pose_estimation.pose_estimator import PoseEstimator as LHM_PoseEstimator

        cfg_train = OmegaConf.load(info["config"])
        cfg = OmegaConf.create()
        cfg.model_name = local_ckpt_dir
        cfg.source_size = cfg_train.dataset.source_image_res
        cfg.src_head_size = getattr(cfg_train.dataset, "src_head_size", 112)
        cfg.render_size = cfg_train.dataset.render_image.high
        cfg.motion_video_read_fps = 6

        hf_model_cls = wrap_model_hub(model_dict["human_lrm_sapdino_bh_sd3_5"])
        lhm = hf_model_cls.from_pretrained(cfg.model_name)
        lhm.to("cuda")

        if _ENGINE_STATE["facedetector"] is None:
            fd = LHM_VGGHeadDetector("./pretrained_models/gagatracker/vgghead/vgg_heads_l.trcd", device="cuda")
            fd.to("cuda")
            _ENGINE_STATE["facedetector"] = fd
        if _ENGINE_STATE["pose_estimator"] is None:
            pe = LHM_PoseEstimator("./pretrained_models/human_model_files/", device="cpu")
            pe.to("cuda")
            pe.device = "cuda"
            _ENGINE_STATE["pose_estimator"] = pe

    _ENGINE_STATE["model_id"] = model_id
    _ENGINE_STATE["family"] = family
    _ENGINE_STATE["lhm"] = lhm
    _ENGINE_STATE["cfg"] = cfg

    dt = time.perf_counter() - t0
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    print(f"✅ Loaded {model_id} ({family.upper()}) in {dt:.1f}s (Allocated VRAM: {vram_alloc:.2f} GB)")
    return _ENGINE_STATE


def normalize_reference_images(image_inputs: List[Any], out_dir: str, max_views: int = 8) -> List[str]:
    """Saves 1 to 8 uploaded reference images into out_dir and returns their file paths."""
    os.makedirs(out_dir, exist_ok=True)
    saved_paths: List[str] = []
    for idx, item in enumerate(image_inputs[:max_views]):
        if item is None:
            continue
        src_path = item[0] if isinstance(item, (tuple, list)) else item
        dst_path = os.path.join(out_dir, f"view_{idx:02d}.png")
        if isinstance(src_path, (str, Path)) and os.path.isfile(str(src_path)):
            img = Image.open(str(src_path)).convert("RGB")
            img.save(dst_path)
            saved_paths.append(dst_path)
        elif isinstance(src_path, Image.Image):
            src_path.convert("RGB").save(dst_path)
            saved_paths.append(dst_path)
        elif isinstance(src_path, np.ndarray):
            Image.fromarray(src_path).convert("RGB").save(dst_path)
            saved_paths.append(dst_path)
    if not saved_paths:
        raise ValueError("Please provide at least 1 valid reference character photo (up to 8 views supported).")
    return saved_paths


def run_lhm_avatar_and_animation(
    reference_images: List[Any],
    model_id: str = PRIMARY_MODEL_ID,
    motion_preset: str = "mimo1",
    custom_smplx_dir: str = "",
    render_mode: str = "neural_render",
    max_motion_frames: int = 120,
    render_fps: int = 30,
    export_canonical_ply: bool = True,
    output_name: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Reconstructs a 3D Gaussian Avatar (.ply + .glb preview) from 1-8 reference photos
    and renders an SMPL-X pose-driven animated character video (.mp4).
    """
    clear_vram()
    state = load_lhm_engine(model_id)
    family = state["family"]
    lhm = state["lhm"]
    cfg = state["cfg"]
    pose_estimator = state["pose_estimator"]
    facedetector = state["facedetector"]
    parsingnet = state["parsingnet"]

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    stem = output_name or f"lhm_avatar_{timestamp}"
    out_dir = os.path.join(OUTPUT_BASE_DIR, stem)
    ref_dir = os.path.join(out_dir, "ref_views")
    os.makedirs(out_dir, exist_ok=True)

    saved_refs = normalize_reference_images(reference_images, ref_dir, max_views=8)
    num_views = len(saved_refs)

    # Resolve SMPL-X motion sequence directory
    available_motions = list_available_motions()
    if custom_smplx_dir and os.path.isdir(custom_smplx_dir.strip()):
        motion_seqs_dir = custom_smplx_dir.strip()
        if os.path.isdir(os.path.join(motion_seqs_dir, "smplx_params")):
            motion_seqs_dir = os.path.join(motion_seqs_dir, "smplx_params")
    elif motion_preset in available_motions:
        motion_seqs_dir = available_motions[motion_preset]
    elif available_motions:
        first_key = next(iter(available_motions))
        motion_seqs_dir = available_motions[first_key]
    else:
        raise FileNotFoundError(
            "No SMPL-X motion sequences found! Make sure Step 2 cached and extracted 3DAIGC/LHMPP-Assets (motion_video.tar)."
        )

    ply_path = os.path.join(out_dir, f"{stem}_avatar_3dgs.ply")
    glb_preview_path = os.path.join(out_dir, f"{stem}_preview.glb")
    video_path = os.path.join(out_dir, f"{stem}_animated.mp4")

    t0 = time.perf_counter()

    if family == "lhmpp":
        os.chdir(LHMPP_REPO_DIR)
        from scripts.inference.app_inference import inference_results
        from scripts.inference.to_gs_ply import run_tpose_export

        is_smplx_free = "SMPLX-FREE" in model_id
        # 1. Export Canonical T-Pose 3D Gaussian Splatting (.ply) Avatar
        if export_canonical_ply:
            print(f"🧊 Reconstructing 3D Gaussian Splatting Avatar (.ply) from {num_views} view(s)...")
            run_tpose_export(
                lhm=lhm,
                cfg=cfg,
                pose_estimator=pose_estimator,
                facedetector=facedetector,
                parsingnet=parsingnet,
                ref_imgs_dir=ref_dir,
                output_ply=ply_path,
            )

        # 2. Render SMPL-X Pose-Driven Animated Character Video (.mp4)
        print(f"🎬 Rendering animated character video ({render_mode}, motion={os.path.basename(os.path.dirname(motion_seqs_dir))})...")
        work_tmp = tempfile.TemporaryDirectory()
        _, rendered_mp4 = inference_results(
            working_dir=work_tmp,
            lhm=lhm,
            cfg=cfg,
            pose_estimator=pose_estimator,
            facedetector=facedetector,
            parsingnet=parsingnet,
            ref_imgs_dir=ref_dir,
            motion_seqs_dir=motion_seqs_dir,
            motion_img_dir=None,
            vis_motion=False,
            motion_img_need_mask=False,
            render_fps=int(render_fps),
            motion_video_read_fps=int(render_fps),
            smplx_free=is_smplx_free,
            render_mode=render_mode,
            motion_size=int(max_motion_frames),
        )
        shutil.copy2(rendered_mp4, video_path)
        work_tmp.cleanup()
    else:
        os.chdir(LHM_REPO_DIR)
        from rembg import remove
        from LHM.runners.infer.utils import prepare_motion_seqs
        from LHM.utils.ffmpeg_utils import images_to_video
        from app import infer_preprocess_image

        image_raw = saved_refs[0]
        with torch.no_grad():
            if parsingnet is not None:
                parsing_out = parsingnet(img_path=image_raw, bbox=None)
                parsing_mask = (parsing_out.masks * 255).astype(np.uint8)
            else:
                img_np = cv2.imread(image_raw)
                parsing_mask = remove(img_np)[..., 3]

            shape_pose = pose_estimator(image_raw)
            image_tensor, _, _ = infer_preprocess_image(
                image_raw,
                mask=parsing_mask,
                intr=None,
                pad_ratio=0,
                bg_color=1.0,
                max_tgt_size=896,
                aspect_standard=5.0 / 3.0,
                enlarge_ratio=[1.0, 1.0],
                render_tgt_size=cfg.source_size,
                multiply=14,
                need_mask=True,
            )

            try:
                rgb_arr = np.array(Image.open(image_raw))[..., :3]
                rgb_t = torch.from_numpy(rgb_arr).permute(2, 0, 1)
                bbox = facedetector.detect_face(rgb_t)
                head_rgb = rgb_t[:, int(bbox[1]) : int(bbox[3]), int(bbox[0]) : int(bbox[2])].permute(1, 2, 0).cpu().numpy()
                src_head_rgb = cv2.resize(head_rgb, dsize=(cfg.src_head_size, cfg.src_head_size), interpolation=cv2.INTER_AREA)
            except Exception as _e:
                src_head_rgb = np.zeros((cfg.src_head_size, cfg.src_head_size, 3), dtype=np.uint8)

            src_head_tensor = torch.from_numpy(src_head_rgb / 255.0).float().permute(2, 0, 1).unsqueeze(0)

            motion_seq = prepare_motion_seqs(
                motion_seqs_dir,
                None,
                save_root=out_dir,
                fps=int(render_fps),
                bg_color=1.0,
                aspect_standard=5.0 / 3.0,
                enlarge_ratio=[1.0, 1.0],
                render_image_res=cfg.render_size,
                multiply=16,
                need_mask=False,
                vis_motion=False,
                motion_size=int(max_motion_frames),
            )

            device = "cuda"
            dtype = torch.float32
            shape_param = torch.tensor(shape_pose.beta, dtype=dtype).unsqueeze(0).to(device)
            lhm.to(dtype)

            smplx_params = {k: v.to(device) for k, v in motion_seq["smplx_params"].items()}
            smplx_params["betas"] = shape_param

            gs_model_list, query_points, transform_mat_neutral_pose = lhm.infer_single_view(
                image_tensor.unsqueeze(0).to(device, dtype),
                src_head_tensor.unsqueeze(0).to(device, dtype),
                None,
                None,
                render_c2ws=motion_seq["render_c2ws"].to(device),
                render_intrs=motion_seq["render_intrs"].to(device),
                render_bg_colors=motion_seq["render_bg_colors"].to(device),
                smplx_params=smplx_params,
            )

            # Export 3DGS .ply avatar for frame 0
            if export_canonical_ply:
                gs_keys = [
                    "root_pose", "body_pose", "jaw_pose", "leye_pose", "reye_pose",
                    "lhand_pose", "rhand_pose", "trans", "focal", "princpt", "img_size_wh", "expr",
                ]
                frame0_smplx = {"betas": shape_param, "transform_mat_neutral_pose": transform_mat_neutral_pose}
                for k in gs_keys:
                    if k in motion_seq["smplx_params"]:
                        frame0_smplx[k] = motion_seq["smplx_params"][k][:, 0:1].to(device)
                output_gs = lhm.animation_infer_gs(gs_model_list, query_points, frame0_smplx)
                output_gs.save_ply(ply_path)

            # Render animated frames
            camera_size = len(motion_seq["motion_seqs"])
            batch_list = []
            batch_size = 24
            keys = [
                "root_pose", "body_pose", "jaw_pose", "leye_pose", "reye_pose",
                "lhand_pose", "rhand_pose", "trans", "focal", "princpt", "img_size_wh", "expr",
            ]
            for batch_i in range(0, camera_size, batch_size):
                batch_smplx = {"betas": shape_param, "transform_mat_neutral_pose": transform_mat_neutral_pose}
                for key in keys:
                    batch_smplx[key] = motion_seq["smplx_params"][key][:, batch_i : batch_i + batch_size].to(device)
                res = lhm.animation_infer(
                    gs_model_list,
                    query_points,
                    batch_smplx,
                    render_c2ws=motion_seq["render_c2ws"][:, batch_i : batch_i + batch_size].to(device),
                    render_intrs=motion_seq["render_intrs"][:, batch_i : batch_i + batch_size].to(device),
                    render_bg_colors=motion_seq["render_bg_colors"][:, batch_i : batch_i + batch_size].to(device),
                )
                comp_rgb = res["comp_rgb"]
                comp_mask = res["comp_mask"]
                comp_mask[comp_mask < 0.5] = 0.0
                batch_rgb = comp_rgb * comp_mask + (1.0 - comp_mask)
                batch_rgb = (batch_rgb.clamp(0, 1) * 255).to(torch.uint8).detach().cpu().numpy()
                batch_list.append(batch_rgb)

            rgb_frames = np.concatenate(batch_list, axis=0)
            images_to_video(rgb_frames, output_path=video_path, fps=int(render_fps), gradio_codec=False, verbose=False)

    # Convert exported .ply to interactive .glb point-cloud preview
    if os.path.isfile(ply_path):
        ply_to_glb_preview(ply_path, glb_preview_path)
    else:
        glb_preview_path = None

    elapsed = time.perf_counter() - t0
    clear_vram()

    summary = (
        f"✅ LHM Avatar & Animation Complete in {elapsed:.1f}s\n"
        f"• Model: {model_id} ({family.upper()})\n"
        f"• Reference Views Used: {num_views} photo(s)\n"
        f"• Motion Sequence: {os.path.basename(os.path.dirname(motion_seqs_dir))} ({render_fps} FPS, up to {max_motion_frames} frames)\n"
        f"• Render Mode: {render_mode if family == 'lhmpp' else 'gs_render'}\n"
        f"• 3D Gaussian Avatar (.ply): {ply_path if os.path.isfile(ply_path) else 'Skipped'}\n"
        f"• Animated Character Video (.mp4): {video_path}"
    )
    print(summary)
    return {
        "video_path": video_path,
        "ply_path": ply_path if os.path.isfile(ply_path) else None,
        "glb_preview_path": glb_preview_path,
        "ref_preview_path": saved_refs[0],
        "summary": summary,
    }


print("✅ Step 3 complete! Core LHM++ & LHM 3D Gaussian Avatar + SMPL-X Animation engine is ready.")

In [ ]:
# @title 🎨 Step 4: Launch Gradio Interactive Web UI (1–8 Reference Views → 3DGS `.ply` Avatar + Animated `.mp4`)
# @markdown Launches the Gradio interface with support for **1 to 8 reference character photos**, **preset or custom SMPL-X driving motions**, **LHM++ (`3DAIGC/LHMPP-700M`) & LHM (`3DAIGC/LHM-1B`) model switching**, **interactive 3D `.glb` avatar preview**, and downloadable **`.ply` + `.mp4`** outputs.
GRADIO_SHARE = True  # @param {type:"boolean"}
GRADIO_PORT = 7860  # @param {type:"integer"}

import os
import gradio as gr
from IPython.display import clear_output

AVAILABLE_MOTIONS = list_available_motions()
MOTION_CHOICES = list(AVAILABLE_MOTIONS.keys()) if AVAILABLE_MOTIONS else ["mimo1", "mimo2", "mimo4", "mimo6"]
EXAMPLE_CASES = list_example_character_cases()
EXAMPLE_CASE_CHOICES = ["(Use Uploaded Photos)"] + list(EXAMPLE_CASES.keys())


def _ui_generate_avatar_and_video(
    primary_photo,
    extra_gallery,
    example_case_name,
    model_id,
    motion_preset,
    custom_smplx_dir,
    render_mode,
    max_motion_frames,
    render_fps,
    export_canonical_ply,
):
    try:
        refs = []
        if example_case_name and example_case_name != "(Use Uploaded Photos)" and example_case_name in EXAMPLE_CASES:
            refs.extend(EXAMPLE_CASES[example_case_name])
        else:
            if primary_photo is not None:
                refs.append(primary_photo)
            if extra_gallery:
                for item in extra_gallery:
                    refs.append(item[0] if isinstance(item, (tuple, list)) else item)

        if not refs:
            raise gr.Error("Please upload at least 1 reference character photo (or choose a built-in example character case).")

        res = run_lhm_avatar_and_animation(
            reference_images=refs,
            model_id=model_id,
            motion_preset=motion_preset,
            custom_smplx_dir=custom_smplx_dir,
            render_mode=render_mode,
            max_motion_frames=int(max_motion_frames),
            render_fps=int(render_fps),
            export_canonical_ply=bool(export_canonical_ply),
        )
        return (
            res["video_path"],
            res["glb_preview_path"],
            res["ply_path"],
            res["video_path"],
            res["summary"],
        )
    except Exception as e:
        raise gr.Error(f"LHM Avatar Reconstruction / Animation failed: {e}")


def _on_ui_load():
    gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return (
        f"🟢 Ready on **{gpu_label}** | Default Model: `{PRIMARY_MODEL_ID}` | "
        f"Discovered **{len(MOTION_CHOICES)}** SMPL-X motion sequences | Cache: `{DRIVE_CACHE_DIR}`"
    )


with gr.Blocks(title="LHM & LHM++ — Animatable 3D Gaussian Avatar Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🕺 LHM & LHM++ — Large Animatable Human Reconstruction Studio\n"
        "Reconstruct **animatable 3D Gaussian Splatting (`.ply`) avatars** and render **SMPL-X pose-driven `.mp4` videos** "
        "from **1 to 8 uncalibrated character photos**."
    )
    status_banner = gr.Markdown("🔄 Initializing LHM Studio...")

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 📸 1. Reference Character Input (1 to 8 Views)")
            in_primary_photo = gr.Image(
                label="Primary Reference Character Photo (View 1)",
                type="filepath",
                height=300,
            )
            in_extra_gallery = gr.Gallery(
                label="Optional Additional Views (Views 2–8 for LHM++ Multi-View Fusion)",
                type="filepath",
                columns=4,
                height=160,
                interactive=True,
            )
            in_example_case = gr.Dropdown(
                choices=EXAMPLE_CASE_CHOICES,
                value=EXAMPLE_CASE_CHOICES[1] if len(EXAMPLE_CASE_CHOICES) > 1 else "(Use Uploaded Photos)",
                label="Or Select Built-in Example Character Set",
                info="Choose '(Use Uploaded Photos)' to use your uploaded photo(s), or pick an official single-view / multi-view test character.",
            )

            gr.Markdown("### 🎞️ 2. Model & SMPL-X Driving Motion Settings")
            in_model_id = gr.Dropdown(
                choices=list(MODEL_REGISTRY.keys()),
                value=PRIMARY_MODEL_ID,
                label="LHM++ / LHM Model Checkpoint",
                info="3DAIGC/LHMPP-700M supports 1–8 reference views on 8 GB VRAM (~0.8s). 3DAIGC/LHM-1B is the 1B single-view model.",
            )
            with gr.Row():
                in_motion_preset = gr.Dropdown(
                    choices=MOTION_CHOICES,
                    value=MOTION_CHOICES[0],
                    label="Built-in SMPL-X Driving Motion Sequence",
                    info="Select a pre-extracted SMPL-X motion sequence from 3DAIGC/LHMPP-Assets (motion_video.tar).",
                )
                in_render_mode = gr.Radio(
                    choices=["neural_render", "gs_render"],
                    value="neural_render",
                    label="LHM++ Render Mode",
                    info="'neural_render' uses the neural refinement decoder for crisp garment/face detail; 'gs_render' rasterizes 3D Gaussians directly.",
                )
            in_custom_smplx_dir = gr.Textbox(
                value="",
                placeholder="/content/drive/MyDrive/my_motion/smplx_params (optional)",
                label="Custom SMPL-X Motion Directory (Optional)",
                info="Optional path to a custom folder of SMPL-X per-frame .json/.npz parameters. Overrides the preset dropdown when set.",
            )
            with gr.Row():
                in_max_frames = gr.Slider(
                    minimum=15,
                    maximum=600,
                    step=15,
                    value=120,
                    label="Max Animation Frames",
                    info="Maximum number of SMPL-X motion frames to render into the output .mp4 video.",
                )
                in_fps = gr.Slider(
                    minimum=15,
                    maximum=60,
                    step=5,
                    value=30,
                    label="Output Video FPS",
                    info="Frame rate for the rendered animated character .mp4 video.",
                )
            in_export_ply = gr.Checkbox(
                value=True,
                label="Export Canonical 3D Gaussian Splatting Avatar (.ply + .glb Preview)",
                info="Exports the reconstructed 3D Gaussian Avatar as a standard .ply splat file and generates an interactive .glb preview.",
            )
            btn_generate = gr.Button("🚀 Reconstruct 3D Avatar & Render Animation", variant="primary")

        with gr.Column(scale=6):
            out_video = gr.Video(
                label="Rendered Animated Character Video (.mp4)",
                autoplay=True,
                height=380,
            )
            out_model3d = gr.Model3D(
                label="Interactive 3D Avatar Point-Cloud Preview (.glb)",
                height=320,
            )
            with gr.Row():
                out_ply_file = gr.File(label="Download 3D Gaussian Avatar (.ply)")
                out_mp4_file = gr.File(label="Download Animated Video (.mp4)")
            out_log = gr.Textbox(
                label="Execution Summary & Diagnostics",
                lines=6,
                interactive=False,
                info="Displays timing, view count, SMPL-X motion sequence, and saved file paths.",
            )

    btn_generate.click(
        fn=_ui_generate_avatar_and_video,
        inputs=[
            in_primary_photo,
            in_extra_gallery,
            in_example_case,
            in_model_id,
            in_motion_preset,
            in_custom_smplx_dir,
            in_render_mode,
            in_max_frames,
            in_fps,
            in_export_ply,
        ],
        outputs=[out_video, out_model3d, out_ply_file, out_mp4_file, out_log],
        concurrency_limit=1,
    )

    demo.load(fn=_on_ui_load, inputs=None, outputs=[status_banner])

clear_output(wait=True)
demo.queue(concurrency_limit=1).launch(
    share=GRADIO_SHARE,
    server_port=GRADIO_PORT,
    inline=False,
    show_error=True,
)

In [ ]:
# @title 🔄 Step 5: Colab Session Keep-Alive (Optional Background Heartbeat)
# @markdown Starts a lightweight background daemon thread that logs GPU memory status periodically so long animation renders or batch jobs are less likely to time out.
ENABLE_KEEPALIVE = True  # @param {type:"boolean"}
HEARTBEAT_INTERVAL_SEC = 180  # @param {type:"integer"}

import threading
import time
import torch

_KEEPALIVE_STOP = False


def _keepalive_worker(interval_sec: int):
    while not _KEEPALIVE_STOP:
        time.sleep(interval_sec)
        if torch.cuda.is_available():
            alloc_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
            reserv_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)
            print(f"[KeepAlive] GPU active — Allocated: {alloc_gb:.2f} GB | Reserved: {reserv_gb:.2f} GB")


if ENABLE_KEEPALIVE:
    _ka_thread = threading.Thread(target=_keepalive_worker, args=(HEARTBEAT_INTERVAL_SEC,), daemon=True)
    _ka_thread.start()
    print(f"✅ Keep-alive daemon started (interval={HEARTBEAT_INTERVAL_SEC}s).")
else:
    print("ℹ️ Keep-alive daemon disabled.")

In [ ]:
# @title ⚡ Step 6: Quick Test (Single/Multi-View Photo → `.ply` 3DGS Avatar + `.mp4` Animation + `FileLink`)
# @markdown Runs a quick head-to-toe test on a custom character photo (or built-in multi-view example character) and renders direct clickable download links (`IPython.display.FileLink`) for the `.ply` 3D Gaussian Avatar and `.mp4` animated character video.
QUICKTEST_IMAGE_PATH = ""  # @param {type:"string"}
QUICKTEST_MODEL_ID = "3DAIGC/LHMPP-700M"  # @param ["3DAIGC/LHMPP-700M", "Damo_XR_Lab/LHMPP-700M-PixelShuffle", "Damo_XR_Lab/LHMPP-700M-SMPLX-FREE", "3DAIGC/LHM-1B", "3DAIGC/LHM-1B-HF", "3DAIGC/LHM-500M-HF", "3DAIGC/LHM-MINI"]
QUICKTEST_MOTION = "mimo1"  # @param {type:"string"}
QUICKTEST_RENDER_MODE = "neural_render"  # @param ["neural_render", "gs_render"]
QUICKTEST_MAX_FRAMES = 60  # @param {type:"slider", min:15, max:300, step:15}

import os
from IPython.display import display, FileLink

if QUICKTEST_IMAGE_PATH.strip() and os.path.exists(QUICKTEST_IMAGE_PATH.strip()):
    p = QUICKTEST_IMAGE_PATH.strip()
    if os.path.isdir(p):
        qt_refs = sorted(
            os.path.join(p, f)
            for f in os.listdir(p)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
        )
    else:
        qt_refs = [p]
else:
    example_cases = list_example_character_cases()
    if example_cases:
        first_case_name, qt_refs = next(iter(example_cases.items()))
        print(f"ℹ️ Using built-in example character set: {first_case_name} ({len(qt_refs)} view(s))")
    else:
        raise FileNotFoundError("Please specify QUICKTEST_IMAGE_PATH or run Step 2 to download 3DAIGC/LHMPP-Assets.")

qt_res = run_lhm_avatar_and_animation(
    reference_images=qt_refs,
    model_id=QUICKTEST_MODEL_ID,
    motion_preset=QUICKTEST_MOTION,
    render_mode=QUICKTEST_RENDER_MODE,
    max_motion_frames=int(QUICKTEST_MAX_FRAMES),
    export_canonical_ply=True,
    output_name="quicktest_lhm_avatar",
)

print("\n📥 Direct Download Links:")
for _label, _key in [
    ("Animated Character Video (.mp4)", "video_path"),
    ("3D Gaussian Splatting Avatar (.ply)", "ply_path"),
    ("Interactive 3D Point-Cloud Preview (.glb)", "glb_preview_path"),
]:
    _fpath = qt_res.get(_key)
    if _fpath and os.path.isfile(_fpath):
        print(f"• {_label}: {_fpath}")
        display(FileLink(_fpath))

In [ ]:
# @title 📂 Step 7: Batch Character Avatar Reconstruction & Animation (`BATCH_INPUT_DIR` → `BATCH_OUTPUT_DIR`)
# @markdown Processes a directory of single character images or subdirectories of 1–8 multi-view character photos with per-item error isolation and `tqdm` progress reporting.
BATCH_INPUT_DIR = "/content/LHM-plusplus/assets/example_multi_images"  # @param {type:"string"}
BATCH_OUTPUT_DIR = "/content/drive/MyDrive/AEI_3D_Cache/LHM/outputs/batch"  # @param {type:"string"}
BATCH_MODEL_ID = "3DAIGC/LHMPP-700M"  # @param ["3DAIGC/LHMPP-700M", "Damo_XR_Lab/LHMPP-700M-PixelShuffle", "Damo_XR_Lab/LHMPP-700M-SMPLX-FREE", "3DAIGC/LHM-1B", "3DAIGC/LHM-1B-HF", "3DAIGC/LHM-500M-HF", "3DAIGC/LHM-MINI"]
BATCH_MOTION_PRESET = "mimo1"  # @param {type:"string"}
BATCH_RENDER_MODE = "neural_render"  # @param ["neural_render", "gs_render"]
BATCH_MAX_FRAMES = 90  # @param {type:"slider", min:15, max:300, step:15}

import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import display, FileLink

os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

batch_jobs = []
if os.path.isdir(BATCH_INPUT_DIR):
    for entry in sorted(os.listdir(BATCH_INPUT_DIR)):
        full = os.path.join(BATCH_INPUT_DIR, entry)
        if os.path.isdir(full):
            imgs = sorted(
                os.path.join(full, f)
                for f in os.listdir(full)
                if Path(f).suffix.lower() in IMAGE_EXTS
            )
            if imgs:
                batch_jobs.append((entry, imgs))
        elif os.path.isfile(full) and Path(full).suffix.lower() in IMAGE_EXTS:
            batch_jobs.append((Path(full).stem, [full]))

if not batch_jobs:
    print(f"⚠️ No character image files or view folders found in {BATCH_INPUT_DIR}")
else:
    print(f"🚀 Starting LHM batch processing on {len(batch_jobs)} character(s) -> {BATCH_OUTPUT_DIR}")
    batch_results = []
    for char_name, ref_list in tqdm(batch_jobs, desc="Batch LHM Avatars", unit="char"):
        try:
            res = run_lhm_avatar_and_animation(
                reference_images=ref_list,
                model_id=BATCH_MODEL_ID,
                motion_preset=BATCH_MOTION_PRESET,
                render_mode=BATCH_RENDER_MODE,
                max_motion_frames=int(BATCH_MAX_FRAMES),
                export_canonical_ply=True,
                output_name=f"batch_{char_name}",
            )
            dst_sub = os.path.join(BATCH_OUTPUT_DIR, char_name)
            os.makedirs(dst_sub, exist_ok=True)
            for k in ("video_path", "ply_path", "glb_preview_path"):
                src_f = res.get(k)
                if src_f and os.path.isfile(src_f):
                    shutil.copy2(src_f, os.path.join(dst_sub, os.path.basename(src_f)))
            batch_results.append((char_name, "OK", dst_sub))
        except Exception as e:
            print(f"❌ Failed on {char_name}: {e}")
            batch_results.append((char_name, f"ERROR: {e}", None))
            clear_vram()

    print("\n📊 Batch Summary:")
    for name, status, folder in batch_results:
        print(f"  • {name}: {status}" + (f" -> {folder}" if folder else ""))